<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Creating FABnet IPv4 Network: Automatic Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** This notebook walks you through creating a FABnet IPv4 (Layer 3) network that connects two nodes on **different** FABRIC sites using **automatic** IP configuration. FABlib assigns IP addresses and configures routes for you during the post-boot phase, so your nodes are ready to communicate as soon as the slice becomes active.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Explain what FABnet IPv4 is and how it provides Layer 3 connectivity across FABRIC sites
2. Create an L3 network with `add_l3network()` using the `IPv4` type
3. Use **automatic configuration** (`set_mode('auto')`) so FABlib assigns IPs and configures interfaces
4. Add routes so nodes on different sites can reach each other through the FABnet backbone
5. Verify cross-site connectivity with `ping`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be comfortable creating basic slices (see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb))

**Tip -- Auto vs. Manual vs. Full Auto:** FABRIC offers three configuration approaches for FABnet:
- **Auto** (this notebook): You create the networks and interfaces yourself, but FABlib assigns IPs and configures them automatically
- **Manual** ([manual notebook](./create_l3network_fabnet_ipv4_manual.ipynb)): You create everything and assign IPs yourself after the slice is active
- **Full Auto** ([full auto notebook](./create_l3network_fabnet_ipv4_full_auto.ipynb)): You call `node.add_fabnet()` and FABlib handles everything

</div>

## Background: What is FABnet IPv4?

FABRIC provides a pair of Layer 3 networking services at every site: **FABnetv4** (IPv4) and **FABnetv6** (IPv6). Think of FABnet as a **private internet** that connects your experiment nodes across the testbed using FABRIC's high-performance backbone links.


**Key concepts:**
- Each site gets its own L3 network with its own subnet (assigned by FABRIC)
- A **gateway** at each site routes traffic between subnets through the FABRIC backbone
- You must add a **route** on each node pointing to the FABnet supernet (`fablib.FABNETV4_SUBNET`) via the local gateway
- With **auto mode**, FABlib picks an IP from the assigned subnet and configures the interface during post-boot

### NIC Component Models

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected via FABNetv4.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

Every FABRIC notebook starts by importing the FABlib library and verifying the configuration.

In [ ]:
# Import Python IP address management libraries (used internally by FABlib for auto config)
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

<div class="fab-warning">

**Tip:** If `show_config()` shows missing or incorrect values, go back and re-run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook.

</div>

## Step 2: Define Slice Parameters

We select two **different** random sites so we can demonstrate cross-site Layer 3 connectivity. Each site will host one node and one FABnet IPv4 network.

In [ ]:
# Name for the slice -- change this if you already have a slice with this name
slice_name = 'MySlice'

# Pick two different random FABRIC sites
[site1,site2] = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node and network names
node1_name = 'Node1'
node2_name = 'Node2'

network1_name='net1'
network2_name='net2'

## Step 3: Create the Slice with FABnet IPv4 Networks

This is the core of the notebook. We build the topology in five logical steps:

1. **Create L3 networks** -- one per site, type `IPv4` (tells FABRIC to use FABnet)
2. **Add nodes** with NIC components
3. **Set interface mode to `auto`** -- tells FABlib to assign IPs automatically
4. **Connect interfaces to networks**
5. **Add routes** pointing to the FABnet supernet via the local gateway

<div class="fab-danger">

**Important:** All interfaces attached to a single `l3network` must be on the **same site**. That is why we create two separate networks -- one per site. FABRIC's backbone routing connects them automatically.

</div>

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Create one FABnet IPv4 network per site ---
# type='IPv4' tells FABRIC this is a FABnet IPv4 L3 network
net1 = slice.add_l3network(name=network1_name, type='IPv4')
net2 = slice.add_l3network(name=network2_name, type='IPv4')

# --- Node1 on site1 ---
node1 = slice.add_node(name=node1_name, site=site1)
# Add a NIC_Basic component and get its first (only) interface
iface1 = node1.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Set to 'auto' so FABlib will assign an IP and configure the interface
iface1.set_mode('auto')
# Attach the interface to the FABnet network on site1
net1.add_interface(iface1)
# Add a route: "to reach ANY FABnet IPv4 address, go through net1's gateway"
node1.add_route(subnet=fablib.FABNETV4_SUBNET, next_hop=net1.get_gateway())

# --- Node2 on site2 ---
node2 = slice.add_node(name=node2_name, site=site2)
# Add a NIC and get its interface
iface2  = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
# Set to 'auto' for automatic IP configuration
iface2.set_mode('auto')
# Attach the interface to the FABnet network on site2
net2.add_interface(iface2)
# Add route to reach all FABnet addresses through net2's gateway
node2.add_route(subnet=fablib.FABNETV4_SUBNET, next_hop=net2.get_gateway())

# Submit the slice request to FABRIC -- this blocks until the slice is ready (~3-5 min)
slice.submit();

<div class="fab-success">

**What just happened?** FABRIC provisioned two VMs on different sites, created a FABnet IPv4 network at each site (each with its own subnet), assigned IP addresses to the interfaces, configured the network devices inside the VMs, and added routing table entries -- all automatically.

</div>

## Step 4: Run the Experiment

With automatic configuration, the slice is ready for experimentation as soon as it becomes active. We will verify connectivity by pinging Node2 from Node1 across the FABnet backbone.

<div class="fab-warning">

**Tip:** Automatic configuration works well when saving slices to a file and reinstantiating them. Configuration tasks are stored in the saved slice, reducing the complexity of notebooks and other runtime steps.

</div>

In [ ]:
# Retrieve the slice (useful if you are running this cell separately)
slice = fablib.get_slice(slice_name)

# Get the node objects
node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           

# Get Node2's automatically assigned IP address on its FABnet network
node2_addr = node2.get_interface(network_name=network2_name).get_ip_addr()

# Ping Node2 from Node1 -- this traverses the FABRIC backbone between sites
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

<div class="fab-success">

**Success!** If you see ping replies above, your two nodes on different FABRIC sites are communicating over the FABnet IPv4 backbone. The round-trip time reflects the physical distance between the two sites.

</div>

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Slice stuck in `Configuring` | Site may be busy or down | Try different sites by re-running the `get_random_sites()` cell |
| `ping` fails between nodes | Routes not configured or interface not up | Check `ip addr` and `ip route` on both nodes; verify `set_mode('auto')` was called before submit |
| `No resources available` | Site lacks NIC_Basic capacity | Choose a different site or try `NIC_ConnectX_6` |
| `submit()` times out | Network issue or high demand | Retry with `slice.submit(wait_timeout=600)` for a longer timeout |
| Asymmetric connectivity (one direction works) | Missing route on one node | Ensure both nodes have `add_route(subnet=fablib.FABNETV4_SUBNET, ...)` |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_sites(count)` | Get a list of distinct random site names | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.add_l3network(name, type)` | Add a Layer 3 network to the slice | [add_l3network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l3network) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |
| `node.add_component(model, name)` | Add a NIC or other component to a node | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `node.add_route(subnet, next_hop)` | Add a static route to a node | [add_route](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_route) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `iface.set_mode('auto')` | Enable automatic IP configuration | [set_mode](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.set_mode) |
| `iface.get_ip_addr()` | Get the IP address assigned to an interface | [get_ip_addr](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_ip_addr) |
| `network.add_interface(iface)` | Connect an interface to a network | [add_interface](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.add_interface) |
| `network.get_gateway()` | Get the gateway IP for a FABnet network | [get_gateway](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_gateway) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **FABnet IPv4 Manual** | [create_l3network_fabnet_ipv4_manual](./create_l3network_fabnet_ipv4_manual.ipynb) | Assign IPs and routes yourself for full control |
| **FABnet IPv4 Full Auto** | [create_l3network_fabnet_ipv4_full_auto](./create_l3network_fabnet_ipv4_full_auto.ipynb) | Use `add_fabnet()` for the simplest possible setup |
| **L2 Local Network** | [create_l2network_basic](../create_l2network_basic/create_l2network_basic_auto.ipynb) | Create a Layer 2 Ethernet on a single site |
| **L2 Wide-Area Network** | [create_l2network_wide_area](../create_l2network_wide_area/create_l2network_wide_area_auto.ipynb) | Create a Layer 2 circuit spanning two sites |
| **FABnet IPv6** | [create_l3network_fabnet_ipv6](../create_l3network_fabnet_ipv6/create_l3network_fabnet_ipv6_auto.ipynb) | Use FABnet with IPv6 addressing |